In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder_path = "/content/drive/MyDrive/data"

import os
files = os.listdir(folder_path)
print(files)  # لیست فایل‌ها

['justforfun_persian.pdf', 'html']


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from langchain_community.document_loaders import PyPDFium2Loader

def minimal_persian_fix(text):
    """
    فقط کاراکترهای مشکل‌دار را تصحیح می‌کند
    فاصله‌ها و ساختار اصلی دست نخورده باقی می‌ماند
    """

    # تصحیح کاراکترهای اشتباه
    char_fixes = {
        'ͷ': 'ک',  # کاراکتر اشتباه به جای ک
        'ͺ': 'ک',  # کاراکتر اشتباه دیگر
        'ͽ': 'گ',  # کاراکتر اشتباه به جای گ
        'ؤ': 'و',  # و همزه‌دار عربی به فارسی
        'ى': 'ی',  # ی عربی به فارسی
        'ة': 'ه',  # ته مربوطه عربی به فارسی
        'ك': 'ک',  # کاف عربی به فارسی
        'ي': 'ی',  # یا عربی به فارسی
    }

    # جایگزینی کاراکترها
    for wrong, correct in char_fixes.items():
        text = text.replace(wrong, correct)

    # تصحیح مشکل خاص chr(876) + فاصله
    text = text.replace(chr(876) + ' ', 'ی ')
    text = text.replace(chr(876) + chr(13), 'ی ')
    text = text.replace(chr(876), 'ی')  # برای موارد دیگر chr(876)

    return text

# بارگذاری PDF
pdf_path = "/content/drive/MyDrive/data/justforfun_persian.pdf"
loader = PyPDFium2Loader(pdf_path)
pdf_docs = loader.load()

# تصحیح محتوای هر صفحه (ساختار pdf_docs حفظ می‌شود)
for i, doc in enumerate(pdf_docs):
    # فقط page_content را تصحیح می‌کنیم
    original_content = doc.page_content
    corrected_content = minimal_persian_fix(original_content)

    # به‌روزرسانی page_content
    doc.page_content = corrected_content

#print(pdf_docs[2].page_content)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader("https://linuxbook.ir/all.html")
web_docs = web_loader.load()

In [ ]:
!pip install wikipedia

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
import time
import random

# لیست تایتل‌های ویکی‌پدیا
wiki_titles = ['ریچارد استالمن', 'لینوس توروالدز', 'لینوکس', 'پروژه گنو', 'نرم‌افزار آزاد', 'بنیاد نرم‌افزار آزاد']

def load_wikipedia_pages_safe(titles, max_retries=3):
    """
    بارگذاری صفحات ویکی‌پدیا با مدیریت خطا و تاخیر
    """
    all_docs = []

    for i, title in enumerate(titles):
        print(f"در حال بارگذاری {i+1}/{len(titles)}: {title}")

        # تلاش مجدد در صورت خطا
        for attempt in range(max_retries):
            try:
                # ایجاد loader برای هر تایتل
                loader = WikipediaLoader(
                    query=title,
                    load_max_docs=1,
                    lang='fa'
                )

                # بارگذاری صفحه
                docs = loader.load()

                if docs:
                    all_docs.extend(docs)
                    print(f"✅ بارگذاری موفق: {title}")
                    break
                else:
                    print(f"⚠️ صفحه‌ای یافت نشد برای: {title}")
                    break

            except Exception as e:
                print(f"❌ خطا در بارگذاری {title} (تلاش {attempt+1}): {str(e)}")

                if attempt < max_retries - 1:
                    # تاخیر تصادفی قبل از تلاش مجدد
                    sleep_time = random.uniform(3, 7)
                    print(f"⏳ صبر {sleep_time:.1f} ثانیه قبل از تلاش مجدد...")
                    time.sleep(sleep_time)
                else:
                    print(f"💥 نتوانست {title} را بارگذاری کند پس از {max_retries} تلاش")

        # تاخیر بین درخواست‌ها برای جلوگیری از Rate Limiting
        if i < len(titles) - 1:  # برای آخرین تایتل تاخیر نیاز نیست
            sleep_time = random.uniform(2, 5)
            print(f"⏳ صبر {sleep_time:.1f} ثانیه قبل از درخواست بعدی...")
            time.sleep(sleep_time)

    return all_docs

# بارگذاری صفحات ویکی‌پدیا
print("🚀 شروع بارگذاری صفحات ویکی‌پدیا...")
print("="*50)

wiki_docs = load_wikipedia_pages_safe(wiki_titles)

print("="*50)
print(f"🎉 بارگذاری کامل شد!")
print(f"📊 تعداد کل صفحات بارگذاری شده: {len(wiki_docs)}")

🚀 شروع بارگذاری صفحات ویکی‌پدیا...
در حال بارگذاری 1/6: ریچارد استالمن
✅ بارگذاری موفق: ریچارد استالمن
⏳ صبر 2.4 ثانیه قبل از درخواست بعدی...
در حال بارگذاری 2/6: لینوس توروالدز
✅ بارگذاری موفق: لینوس توروالدز
⏳ صبر 4.3 ثانیه قبل از درخواست بعدی...
در حال بارگذاری 3/6: لینوکس
✅ بارگذاری موفق: لینوکس
⏳ صبر 4.1 ثانیه قبل از درخواست بعدی...
در حال بارگذاری 4/6: پروژه گنو
✅ بارگذاری موفق: پروژه گنو
⏳ صبر 3.5 ثانیه قبل از درخواست بعدی...
در حال بارگذاری 5/6: نرم‌افزار آزاد
✅ بارگذاری موفق: نرم‌افزار آزاد
⏳ صبر 2.2 ثانیه قبل از درخواست بعدی...
در حال بارگذاری 6/6: بنیاد نرم‌افزار آزاد
✅ بارگذاری موفق: بنیاد نرم‌افزار آزاد
🎉 بارگذاری کامل شد!
📊 تعداد کل صفحات بارگذاری شده: 6


In [ ]:
print("Number of wikipedia pages (loaded Document objects):", len(wiki_docs))

Number of wikipedia pages (loaded Document objects): 6


In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import BSHTMLLoader
import os

# مسیر پوشه HTML
html_directory = "/content/drive/MyDrive/data/html"  # یا مسیر دقیق پوشه شما

def load_html_files(directory_path):
    """
    بارگذاری فایل‌های HTML از یک پوشه با استفاده از DirectoryLoader و BSHTMLLoader
    """
    try:
        # بررسی وجود پوشه
        if not os.path.exists(directory_path):
            print(f"❌ پوشه یافت نشد: {directory_path}")
            return []

        print(f"📁 در حال بررسی پوشه: {directory_path}")

        # شمارش فایل‌های HTML
        html_files = [f for f in os.listdir(directory_path) if f.endswith('.html')]
        print(f"📄 تعداد فایل‌های HTML یافت شده: {len(html_files)}")

        if len(html_files) == 0:
            print("⚠️ هیچ فایل HTML در پوشه یافت نشد!")
            return []

        # نمایش چند فایل اول
        print("📋 نمونه فایل‌ها:")
        for i, file in enumerate(html_files[:5]):
            print(f"  {i+1}. {file}")
        if len(html_files) > 5:
            print(f"  ... و {len(html_files) - 5} فایل دیگر")

        print("\n🚀 شروع بارگذاری فایل‌ها...")

        # ایجاد DirectoryLoader با BSHTMLLoader
        loader = DirectoryLoader(
            path=directory_path,
            glob="*.html",  # فقط فایل‌های HTML
            loader_cls=BSHTMLLoader,  # استفاده از BeautifulSoup برای پارس HTML
            show_progress=True,  # نمایش پیشرفت
            use_multithreading=True  # استفاده از چندنخی برای سرعت بیشتر
        )

        # بارگذاری فایل‌ها
        docs = loader.load()

        print(f"✅ بارگذاری کامل شد!")
        print(f"📊 تعداد documents بارگذاری شده: {len(docs)}")

        return docs

    except Exception as e:
        print(f"❌ خطا در بارگذاری فایل‌ها: {str(e)}")
        return []

# بارگذاری فایل‌های HTML
print("=" * 50)
print("📂 بارگذاری فایل‌های HTML از وب‌سایت استالمن")
print("=" * 50)

html_docs = load_html_files(html_directory)

# نمایش نتایج
if html_docs and len(html_docs) > 0:
    print("\n📋 اطلاعات documents بارگذاری شده:")
    print("-" * 30)

    for i, doc in enumerate(html_docs[:3]):  # نمایش ۳ مورد اول
        # استخراج نام فایل از metadata
        source = doc.metadata.get('source', 'نامشخص')
        filename = os.path.basename(source)
        content_length = len(doc.page_content)

        print(f"{i+1}. فایل: {filename}")
        print(f"   طول محتوا: {content_length:,} کاراکتر")
        print(f"   مسیر: {source}")
        print()

    if len(html_docs) > 3:
        print(f"... و {len(html_docs) - 3} document دیگر")

    # نمایش نمونه محتوا
    print(f"\n📄 نمونه محتوای اولین فایل:")
    print("-" * 30)
    first_doc = html_docs[0]
    print(f"عنوان: {first_doc.metadata.get('title', 'بدون عنوان')}")
    print(f"محتوا: {first_doc.page_content[:200]}...")

    # نمایش metadata کامل
    print(f"\n🏷️ metadata اولین document:")
    print(first_doc.metadata)

else:
    print("❌ هیچ فایل HTML بارگذاری نشد!")

# بررسی ساختار داده
print(f"\n🔍 ساختار داده:")
print(f"نوع html_docs: {type(html_docs)}")
if html_docs:
    print(f"نوع هر document: {type(html_docs[0])}")
    print(f"خصوصیات document: {[attr for attr in dir(html_docs[0]) if not attr.startswith('_')]}")

# آمار کلی
if html_docs:
    total_chars = sum(len(doc.page_content) for doc in html_docs)
    avg_chars = total_chars / len(html_docs)
    print(f"\n📊 آمار کلی:")
    print(f"تعداد کل documents: {len(html_docs)}")
    print(f"مجموع کاراکترها: {total_chars:,}")
    print(f"میانگین طول هر document: {avg_chars:,.0f} کاراکتر")

📂 بارگذاری فایل‌های HTML از وب‌سایت استالمن
📁 در حال بررسی پوشه: /content/drive/MyDrive/data/html
📄 تعداد فایل‌های HTML یافت شده: 179
📋 نمونه فایل‌ها:
  1. articles_childhood-sweetheart.html
  2. lyft.html
  3. photos_china_2000_RMS_in_China_2000.html
  4. articles_jinnetic.ru.html
  5. pay-toilets.html
  ... و 174 فایل دیگر

🚀 شروع بارگذاری فایل‌ها...


100%|██████████| 179/179 [00:03<00:00, 54.66it/s]

✅ بارگذاری کامل شد!
📊 تعداد documents بارگذاری شده: 179

📋 اطلاعات documents بارگذاری شده:
------------------------------
1. فایل: articles_childhood-sweetheart.html
   طول محتوا: 6,330 کاراکتر
   مسیر: /content/drive/MyDrive/data/html/articles_childhood-sweetheart.html

2. فایل: lyft.html
   طول محتوا: 2,995 کاراکتر
   مسیر: /content/drive/MyDrive/data/html/lyft.html

3. فایل: photos_china_2000_RMS_in_China_2000.html
   طول محتوا: 1,964 کاراکتر
   مسیر: /content/drive/MyDrive/data/html/photos_china_2000_RMS_in_China_2000.html

... و 176 document دیگر

📄 نمونه محتوای اولین فایل:
------------------------------
عنوان: My Childhood Sweetheart
محتوا: 

My Childhood Sweetheart





Richard Stallman's personal site.
https://stallman.org

For current political commentary, see
the daily
political notes.


RMS's Bio |
The GNU Project


My Childhood Swe...

🏷️ metadata اولین document:
{'source': '/content/drive/MyDrive/data/html/articles_childhood-sweetheart.html', 'title': 'My Childhood Sweethe

In [ ]:
print("Number of pages (loaded Document objects):", len(html_docs))

Number of pages (loaded Document objects): 179


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter

def combine_all_documents(*doc_lists):
    """
    ترکیب تمام اسناد از منابع مختلف در یک لیست واحد
    """
    all_docs = []
    sources = ['PDF', 'Web', 'Wikipedia', 'HTML']

    for i, docs in enumerate(doc_lists):
        if docs is not None and len(docs) > 0:
            source_name = sources[i] if i < len(sources) else f'Source_{i+1}'
            print(f"📚 {source_name}: {len(docs)} documents")

            # اضافه کردن نام منبع به metadata
            for doc in docs:
                if 'source_type' not in doc.metadata:
                    doc.metadata['source_type'] = source_name

            all_docs.extend(docs)
        else:
            source_name = sources[i] if i < len(sources) else f'Source_{i+1}'
            print(f"❌ {source_name}: هیچ سندی یافت نشد")

    return all_docs

def split_documents_advanced(documents, chunk_size=1000, chunk_overlap=200):
    """
    جداسازی پیشرفته اسناد با در نظر گیری نوع محتوا
    """

    # تنظیمات مختلف برای انواع مختلف محتوا
    splitters = {
        'default': RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
        ),
        'persian': RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", "۔ ", "؟ ", "! ", "؟", "۔", " ", ""]
        ),
        'html': RecursiveCharacterTextSplitter(
            chunk_size=chunk_size * 2,  # HTML معمولاً طولانی‌تر است
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
    }

    all_chunks = []
    stats = {
        'original_docs': len(documents),
        'total_chunks': 0,
        'by_source': {}
    }

    for doc in documents:
        source_type = doc.metadata.get('source_type', 'Unknown')

        # انتخاب splitter مناسب
        if source_type == 'HTML':
            splitter = splitters['html']
        elif source_type in ['PDF', 'Wikipedia'] and 'فارسی' in doc.page_content[:100]:
            splitter = splitters['persian']
        else:
            splitter = splitters['default']

        # جداسازی سند
        chunks = splitter.split_documents([doc])

        # اضافه کردن اطلاعات chunk به metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata['chunk_index'] = i
            chunk.metadata['total_chunks'] = len(chunks)
            chunk.metadata['original_doc_length'] = len(doc.page_content)

        all_chunks.extend(chunks)

        # آپدیت آمار
        if source_type not in stats['by_source']:
            stats['by_source'][source_type] = {'docs': 0, 'chunks': 0}

        stats['by_source'][source_type]['docs'] += 1
        stats['by_source'][source_type]['chunks'] += len(chunks)

    stats['total_chunks'] = len(all_chunks)
    return all_chunks, stats

def analyze_documents(documents):
    """
    تحلیل اسناد قبل از جداسازی
    """
    if not documents:
        print("❌ هیچ سندی برای تحلیل یافت نشد!")
        return

    print("📊 تحلیل اسناد:")
    print("-" * 40)

    total_length = sum(len(doc.page_content) for doc in documents)
    avg_length = total_length / len(documents)

    # یافتن طولانی‌ترین و کوتاه‌ترین سند
    lengths = [len(doc.page_content) for doc in documents]
    max_length = max(lengths)
    min_length = min(lengths)

    print(f"تعداد کل اسناد: {len(documents)}")
    print(f"مجموع کاراکترها: {total_length:,}")
    print(f"میانگین طول: {avg_length:,.0f} کاراکتر")
    print(f"طولانی‌ترین سند: {max_length:,} کاراکتر")
    print(f"کوتاه‌ترین سند: {min_length:,} کاراکتر")

    # تحلیل بر اساس نوع منبع
    source_stats = {}
    for doc in documents:
        source = doc.metadata.get('source_type', 'Unknown')
        if source not in source_stats:
            source_stats[source] = []
        source_stats[source].append(len(doc.page_content))

    print(f"\n📋 آمار بر اساس منبع:")
    for source, lengths in source_stats.items():
        avg_len = sum(lengths) / len(lengths)
        print(f"  {source}: {len(lengths)} سند، میانگین {avg_len:,.0f} کاراکتر")

# ترکیب تمام اسناد
print("🔄 ترکیب اسناد از منابع مختلف...")
print("=" * 50)

all_documents = combine_all_documents(pdf_docs, web_docs, wiki_docs, html_docs)

print(f"\n📊 مجموع اسناد ترکیب شده: {len(all_documents)}")

# تحلیل اسناد قبل از جداسازی
analyze_documents(all_documents)

# جداسازی اسناد
print("\n🔪 شروع جداسازی اسناد...")
print("=" * 50)

# تنظیمات جداسازی
CHUNK_SIZE = 1000      # اندازه هر قطعه
CHUNK_OVERLAP = 200    # همپوشانی بین قطعات

splitted_docs, splitting_stats = split_documents_advanced(
    all_documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# نمایش نتایج
print("✅ جداسازی کامل شد!")
print("=" * 50)

print(f"📊 آمار جداسازی:")
print(f"تعداد اسناد اصلی: {splitting_stats['original_docs']}")
print(f"تعداد قطعات ایجاد شده: {splitting_stats['total_chunks']}")
print(f"نسبت تبدیل: {splitting_stats['total_chunks'] / splitting_stats['original_docs']:.1f} قطعه به ازای هر سند")

print(f"\n📋 آمار بر اساس منبع:")
for source, stats in splitting_stats['by_source'].items():
    ratio = stats['chunks'] / stats['docs']
    print(f"  {source}: {stats['docs']} سند → {stats['chunks']} قطعه (نسبت: {ratio:.1f})")

# نمایش نمونه قطعات
print(f"\n📄 نمونه قطعات:")
print("-" * 30)

for i, chunk in enumerate(splitted_docs[:3]):
    source_type = chunk.metadata.get('source_type', 'Unknown')
    chunk_index = chunk.metadata.get('chunk_index', 'N/A')
    total_chunks = chunk.metadata.get('total_chunks', 'N/A')

    print(f"قطعه {i+1}:")
    print(f"  منبع: {source_type}")
    print(f"  شماره قطعه: {chunk_index + 1}/{total_chunks}")
    print(f"  طول: {len(chunk.page_content)} کاراکتر")
    print(f"  محتوا: {chunk.page_content[:100]}...")
    print()

# خروجی نهایی برای کد اصلی
print('The number of splitted documents:', len(splitted_docs))

# بررسی کیفیت جداسازی
if splitted_docs:
    chunk_lengths = [len(chunk.page_content) for chunk in splitted_docs]
    avg_chunk_length = sum(chunk_lengths) / len(chunk_lengths)
    max_chunk_length = max(chunk_lengths)
    min_chunk_length = min(chunk_lengths)

    print(f"\n🎯 کیفیت جداسازی:")
    print(f"میانگین طول قطعات: {avg_chunk_length:.0f} کاراکتر")
    print(f"طولانی‌ترین قطعه: {max_chunk_length} کاراکتر")
    print(f"کوتاه‌ترین قطعه: {min_chunk_length} کاراکتر")

🔄 ترکیب اسناد از منابع مختلف...
📚 PDF: 204 documents
📚 Web: 1 documents
📚 Wikipedia: 6 documents
📚 HTML: 179 documents

📊 مجموع اسناد ترکیب شده: 390
📊 تحلیل اسناد:
----------------------------------------
تعداد کل اسناد: 390
مجموع کاراکترها: 2,472,319
میانگین طول: 6,339 کاراکتر
طولانی‌ترین سند: 329,640 کاراکتر
کوتاه‌ترین سند: 11 کاراکتر

📋 آمار بر اساس منبع:
  PDF: 204 سند، میانگین 2,147 کاراکتر
  Web: 1 سند، میانگین 238,830 کاراکتر
  Wikipedia: 6 سند، میانگین 3,650 کاراکتر
  HTML: 179 سند، میانگین 9,908 کاراکتر

🔪 شروع جداسازی اسناد...
✅ جداسازی کامل شد!
📊 آمار جداسازی:
تعداد اسناد اصلی: 390
تعداد قطعات ایجاد شده: 2068
نسبت تبدیل: 5.3 قطعه به ازای هر سند

📋 آمار بر اساس منبع:
  PDF: 204 سند → 622 قطعه (نسبت: 3.0)
  Web: 1 سند → 336 قطعه (نسبت: 336.0)
  Wikipedia: 6 سند → 34 قطعه (نسبت: 5.7)
  HTML: 179 سند → 1076 قطعه (نسبت: 6.0)

📄 نمونه قطعات:
------------------------------
قطعه 1:
  منبع: PDF
  شماره قطعه: 1/1
  طول: 121 کاراکتر
  محتوا: فقط برای تفریح
داستان یک انقلابی اتفاقی
لینو

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import torch

# بهترین مدل embedding رایگان چندزبانه
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("🚀 استفاده از multilingual-e5-large (رایگان و قدرتمند)")
print(f"💻 دستگاه: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# ساخت vector store
print("📦 ایجاد vector store...")

vectorstore = Chroma.from_documents(
    documents=splitted_docs,
    embedding=embedding_model,
    persist_directory="/content/drive/MyDrive/chroma_db2"
)

print(f"✅ {len(splitted_docs)} سند به vector store اضافه شد")
print("💾 Vector store در /content/drive/MyDrive/chroma_db2 ذخیره شد")

# تست جستجو
print("\n🔍 تست جستجو:")
test_query = "لینوس توروالدز"
results = vectorstore.similarity_search(test_query, k=3)

for i, doc in enumerate(results):
    source = doc.metadata.get('source_type', 'Unknown')
    print(f"{i+1}. منبع: {source}")
    print(f"   محتوا: {doc.page_content[:150]}...")
    print()

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🚀 استفاده از multilingual-e5-large (رایگان و قدرتمند)
💻 دستگاه: GPU
📦 ایجاد vector store...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✅ 2068 سند به vector store اضافه شد
💾 Vector store در /content/drive/MyDrive/chroma_db2 ذخیره شد

🔍 تست جستجو:
1. منبع: Wikipedia
   محتوا: لینوس بندیکت توروالدز (به سوئدی: Linus Benedict Torvalds؛ آوایش: ˈli:nɵs ˈtu:rvalds، ) (متولد ۲۸ دسامبر ۱۹۶۹ در هلسینکی فنلاند) یک مهندس نرم‌افزار فنل...

2. منبع: Wikipedia
   محتوا: == سال‌های آغازین ==
لینوس توروالدز در ۲۸ دسامبر ۱۹۶۹ (۷ دی ۱۳۴۸) در خانواده‌ای فنلاندی-سوئدی چشم به جهان گشود. پدرش نیلز توروالدز خبرنگار رادیو و تلو...

3. منبع: Wikipedia
   محتوا: == زندگی شخصی ==
توروالدز با تاو مونی ازدواج کرد. توروالدز اولین بار در پاییز ۱۹۹۳ با تاو آشنا شد؛ وی در حال راه‌اندازی یک آزمایشگاه برای تمرینات دانش...



In [ ]:
# تست جستجو
print("\n🔍 تست جستجو:")
test_query = "کالی لینوکس چیست"
results = vectorstore.similarity_search(test_query, k=3)

for i, doc in enumerate(results):
    source = doc.metadata.get('source_type', 'Unknown')
    print(f"{i+1}. منبع: {source}")
    print(f"   محتوا: {doc.page_content[:150]}...")
    print()


🔍 تست جستجو:
1. منبع: Wikipedia
   محتوا: لینوکس، توزیع‌های مختلفی دارد، از جمله دبیان، سنت او اس (مناسب سرور)، کالی لینوکس (که بیشتر برای تست نفوذ استفاده می‌شود) فدورا و نمونه‌های دیگر نام د...

2. منبع: Web
   محتوا: برای بحث بیشتر به این مجموعه کامنت مراجعه کنین 
لینوکس و زندگی. نوشته جادی www.jadi.netبرای دسترسی به آخرین نسخه کتاب در فرمت‌های مختلف و همچنین حمایت...

3. منبع: Web
   محتوا: پیوند به بیرون: وب‌سایت اختصاصی نهاد OLPC 
لینوکس و زندگی. نوشته جادی www.jadi.netبرای دسترسی به آخرین نسخه کتاب در فرمت‌های مختلف و همچنین حمایت مالی...



In [ ]:
!pip install langchain-ollama

In [ ]:
!pip install langchain-cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.2 MB/s eta 0:00:00
  Attempting uninstall: httpx-sse
    Found existing installation: httpx-sse 0.4.1
    Uninstalling httpx-sse-0.4.1:
      Successfully uninstalled httpx-sse-0.4.1


In [ ]:
from langchain_cohere import ChatCohere
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
import os
import time
import re

# تنظیم API key
os.environ["COHERE_API_KEY"] = "BEwyBzkFyOI8NGURnoMxcrEP7N9TY0nGMeq4ro4b"

# مدل Cohere
llm = ChatCohere(
    model="command-r-plus",  # مدل قدرتمند Cohere
    temperature=0,           # پاسخ قطعی
    max_tokens=50           # پاسخ کوتاه
)

# Prompt template ساده
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
بر اساس متن زیر، به سوال پاسخ کوتاه (حداکثر 4 کلمه) بده:

متن: {context}

سوال: {question}

پاسخ کوتاه:"""
)

def extract_answer(text):
    """استخراج پاسخ کوتاه از متن"""
    # حذف کلمات اضافی
    text = text.strip()
    words = text.split()

    # حداکثر 4 کلمه
    if len(words) > 4:
        text = " ".join(words[:4])

    return text

def format_docs(docs):
    """ترکیب اسناد برای context"""
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
rag_chain = (
    {
        "context": vectorstore.as_retriever(search_kwargs={"k": 3}) | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
    | extract_answer
)

def get_answer(question, question_num):
    """دریافت پاسخ با مدیریت rate limiting"""
    try:
        # تاخیر برای جلوگیری از rate limit
        time.sleep(7)

        answer = rag_chain.invoke(question)

        return {
            "question_number": question_num,
            "answer": answer
        }
    except Exception as e:
        print(f"خطا در سوال {question_num}: {e}")
        return {
            "question_number": question_num,
            "answer": "نامشخص"
        }

# پردازش سوالات
questions = [
    "پرسش ۱: توروالدز برای کار در چه موسسه‌ای دانشگاه هلسینکی را ترک گفت؟",
    "پرسش ۲: آندرو تاننباوم استاد کدام دانشگاه است؟",
    "پرسش ۳: در سال ۲۰۰۶ چند درصد از هسته لینوکس توسط توروالدز نوشته شد (به عدد)؟",
    "پرسش ۴: چه کسی بنیاد نرم‌افزارهای آزاد را بنا نهاد؟",
    "پرسش ۵: ریچارد استالمن در ۲۱ سالگی در کدام شرکت کار می‌کرد؟",
    "پرسش ۶: یکی از مشهورترین پروژه‌هایی که در ابتدا پروژه‌ی آزاد و آکادمیک بود اما بعد وارد محیط بسته‌ی تجاری شد چه بود؟",
    "پرسش ۷: لینکدین در سانسور کردن حساب‌ها به درخواست چه کشوری مشهور است؟",
    "پرسش ۸: ریچارد استالمن پیشنهاد می‌کند به‌جای گوگل مپ از چه سرویسی استفاده کنیم؟",
    "پرسش ۹: آزادی صفرم در نرم‌افزار آزاد چه عنوانی دارد؟",
    "پرسش ۱۰: آیا یک نرم‌افزار آزاد لزوماً رایگان است (بله یا خیر)؟",
    "پرسش ۱۱: استاندارد ناظر بر فایل‌ها و دایرکتوری‌ها به‌اختصار چه نامیده می‌شود؟",
    "پرسش ۱۲: اولین ریپلای به ایمیل درخواست کار چیست؟",
    "پرسش ۱۳: اگر امروز که از شنبه ورزش می‌کنم در واقع دچار چه بایاسی شده‌ایم؟",
    "پرسش ۱۴: دنبال یاد گرفتن کدوم یکی باشیم: برنامه‌نویسی یا دستور زبان یک زبان خاص؟",
    "پرسش ۱۵: اگه هدف‌مون اینه که بریم گوگل کار کنیم اول از همه چه‌چیزی رو سرچ کنیم؟",
    "پرسش ۱۶: در بیانیه‌ی هکرها گفته شده که جرم آن‌ها در یک کلمه چیست؟"
]

print("🤖 شروع پردازش سوالات...")

# پردازش تمام سوالات
answers = []
for i, question in enumerate(questions, 1):
    print(f"سوال {i}/16 در حال پردازش...")
    result = get_answer(question, i)
    answers.append(result)
    print(f"پاسخ: {result['answer']}")

# تعریف متغیرهای جداگانه
answer1 = answers[0]
answer2 = answers[1]
answer3 = answers[2]
answer4 = answers[3]
answer5 = answers[4]
answer6 = answers[5]
answer7 = answers[6]
answer8 = answers[7]
answer9 = answers[8]
answer10 = answers[9]
answer11 = answers[10]
answer12 = answers[11]
answer13 = answers[12]
answer14 = answers[13]
answer15 = answers[14]
answer16 = answers[15]

print("\n✅ همه سوالات پردازش شد!")
print("📋 نمونه پاسخ‌ها:")
for ans in answers[:3]:
    print(f"سوال {ans['question_number']}: {ans['answer']}")

🤖 شروع پردازش سوالات...
سوال 1/16 در حال پردازش...
پاسخ: ارتش فنلاند
سوال 2/16 در حال پردازش...
پاسخ: دانشگاه آمستردام
سوال 3/16 در حال پردازش...
پاسخ: دو درصد
سوال 4/16 در حال پردازش...
پاسخ: ریچارد استالمن
سوال 5/16 در حال پردازش...
پاسخ: شرکت IBM
سوال 6/16 در حال پردازش...
پاسخ: لیسپ (LISP)
سوال 7/16 در حال پردازش...
پاسخ: حساب‌های چین
سوال 8/16 در حال پردازش...
پاسخ: اوپن استریت مپ
سوال 9/16 در حال پردازش...
پاسخ: اجرای برنامه
سوال 10/16 در حال پردازش...
پاسخ: خیر
سوال 11/16 در حال پردازش...
پاسخ: FHS
سوال 12/16 در حال پردازش...
پاسخ: رزومه
سوال 13/16 در حال پردازش...
پاسخ: پروژکشن بایاس
سوال 14/16 در حال پردازش...
پاسخ: برنامه‌نویسی
سوال 15/16 در حال پردازش...
پاسخ: "چگونه در گوگل استخدام
سوال 16/16 در حال پردازش...
پاسخ: کنجکاوی

✅ همه سوالات پردازش شد!
📋 نمونه پاسخ‌ها:
سوال 1: ارتش فنلاند
سوال 2: دانشگاه آمستردام
سوال 3: دو درصد
